In [1]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertModel

In [ ]:
BATCH_SIZE = 16
LEARNING_RATE = 2e-5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
MAX_EPOCHS = 4  # Maximum epochs for early stopping
PATIENCE = 3     # Patience for early stopping

tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
ds = load_dataset("KushT/bbc_news_multiclass_train_val_test")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,Chinese wine tempts Italy's Illva Italy's Illv...,0
1,Labour chooses Manchester The Labour Party wil...,2
2,Iran budget seeks state sell-offs Iran's presi...,0
3,Roundabout continues nostalgia trip The new bi...,1
4,US charity anthem is re-released We Are The Wo...,1
...,...,...
1507,Game warnings 'must be clearer' Violent video ...,2
1508,Blair ready to call election Tony Blair seems ...,2
1509,Mourinho expects fight to finish Chelsea manag...,3
1510,India power shares jump on debut Shares in Ind...,0


In [4]:
class MultiClassClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels  # Labels should be integers: 0, 1, 2, ..., num_classes-1
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)  # Changed to scalar tensor of type long
        }

In [5]:
class BertForMultiClassClassification(nn.Module):
    def __init__(self, num_classes):
        super(BertForMultiClassClassification, self).__init__()
        self.bert = BertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output size is num_classes
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits, no sigmoid

In [6]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [7]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, save_path, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    start_train = perf_counter()
    
    # Initialize best metrics
    best_train_acc = 0
    best_train_precisions = None
    best_train_recalls = None
    best_train_f1s = None
    best_val_acc = 0
    best_val_precisions = None
    best_val_recalls = None
    best_val_f1s = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{max_epochs}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size,)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # CrossEntropyLoss expects logits and long labels
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()  # Get class indices
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            loss.backward()
            optimizer.step()
        
        train_loss /= len(train_dataloader)
        train_preds = np.array(train_preds)
        train_true = np.array(train_true)
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        val_preds = np.array(val_preds)
        val_true = np.array(val_true)
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}")
        print(f"Epoch {epoch + 1}/{max_epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_acc = train_acc
            best_train_precisions = train_precisions
            best_train_recalls = train_recalls
            best_train_f1s = train_f1s
            best_val_acc = val_acc
            best_val_precisions = val_precisions
            best_val_recalls = val_recalls
            best_val_f1s = val_f1s
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print("Model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    
    total_train_time = perf_counter() - start_train
    return (best_train_acc, best_train_precisions, best_train_recalls, best_train_f1s,
            best_val_acc, best_val_precisions, best_val_recalls, best_val_f1s, total_train_time)

In [8]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].item()
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = torch.argmax(output, dim=1).item()  # Scalar integer
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [9]:
train_texts = train_df['text'].values
train_labels = train_df['label'].values  # Must be integers: 0, 1, 2, ..., num_classes-1

val_texts = val_df['text'].values
val_labels = val_df['label'].values

test_texts = test_df['text'].values
test_labels = test_df['label'].values

train_dataset = MultiClassClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiClassClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiClassClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

seeds = [2, 3, 5]

results = []

# Get number of classes from the training data
num_classes = train_df['label'].nunique()

# Loop through seeds
for seed in seeds:
    torch.manual_seed(seed)
    model = BertForMultiClassClassification(num_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    save_path = f'results/bert_multiclass2_bs{BATCH_SIZE}_lr{LEARNING_RATE}_seed{seed}.pt'

    # Train
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion, save_path),
         {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}), max_usage=True, retval=True)

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    (train_acc, train_precisions, train_recalls, train_f1s,
     val_acc, val_precisions, val_recalls, val_f1s, total_train_time) = retval

    # Load best model
    model.load_state_dict(torch.load(save_path))

    # Evaluate
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, test_retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
    total_test_time = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    predictions, true_labels = test_retval
    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    # Store individual results for this seed
    results.append({
        'seed': seed,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'train_acc': train_acc,
        'train_precisions': train_precisions.tolist(),
        'train_recalls': train_recalls.tolist(),
        'train_f1s': train_f1s.tolist(),
        'max_memory_usage_train': max_memory_usage_train,
        'max_vram_usage_train': max_vram_usage_train,
        'total_train_time': total_train_time,
        'val_acc': val_acc,
        'val_precisions': val_precisions.tolist(),
        'val_recalls': val_recalls.tolist(),
        'val_f1s': val_f1s.tolist(),
        'test_acc': test_acc,
        'test_precisions': test_precisions.tolist(),
        'test_recalls': test_recalls.tolist(),
        'test_f1s': test_f1s.tolist(),
        'max_memory_usage_test': max_memory_usage_test,
        'max_vram_usage_test': max_vram_usage_test,
        'total_test_time': total_test_time
    })

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
Epoch 1/4:   0%|          | 0/48 [00:00<?, ?it/s]c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch 1/4 - Train Loss: 0.9166, Acc: 0.7956, F1: [0.77579092 0.75764706 0.8122449  0.83544304 0.78040541]
Epoch 1/4 - Val Loss: 0.2482, Acc: 0.9657, F1: [0.96045198 0.94890511 0.95652174 0.99421965 0.96240602]
Model saved!


Epoch 2/4 - Train Loss: 0.1668, Acc: 0.9808, F1: [0.97250362 0.97709924 0.97535211 0.99711816 0.97989031]
Epoch 2/4 - Val Loss: 0.0947, Acc: 0.9789, F1: [0.97142857 0.97058824 0.97142857 1.         0.97744361]
Model saved!


Epoch 3/4 - Train Loss: 0.0749, Acc: 0.9888, F1: [0.97854077 0.99808795 0.975      0.99856115 0.99451554]
Epoch 3/4 - Val Loss: 0.0659, Acc: 0.9842, F1: [0.97701149 0.98507463 0.98611111 0.99421965 0.97744361]
Model saved!


Epoch 4/4 - Train Loss: 0.0353, Acc: 0.9960, F1: [0.99276411 0.99808795 0.99121265 1.         0.99817185]
Epoch 4/4 - Val Loss: 0.0573, Acc: 0.9868, F1: [0.97701149 0.98507463 0.99300699 0.99421965 0.98507463]
Model saved!


...

Epoch 2/4 - Train Loss: 0.1570, Acc: 0.9769, F1: [0.96802326 0.98084291 0.97001764 0.99856115 0.96376812]
Epoch 2/4 - Val Loss: 0.1271, Acc: 0.9657, F1: [0.97109827 0.94202899 0.97931034 1.         0.921875  ]
Model saved!


Epoch 3/4 - Train Loss: 0.0665, Acc: 0.9888, F1: [0.98118669 0.99428571 0.98765432 0.99712644 0.98348624]
Epoch 3/4 - Val Loss: 0.0856, Acc: 0.9763, F1: [0.95454545 0.98484848 0.9787234  0.99421965 0.97058824]
Model saved!


Epoch 4/4 - Train Loss: 0.0311, Acc: 0.9967, F1: [0.99421965 0.99808795 0.99823633 1.         0.99270073]
Epoch 4/4 - Val Loss: 0.0957, Acc: 0.9736, F1: [0.96       0.97014925 0.97101449 1.         0.96350365]


C:\Users\Rafael\AppData\Local\Temp\ipykernel_268\1483028676.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path))
c:\Users\Rafael

Test Time: 5.00 seconds
Test Metrics:
Accuracy: 0.9880239520958084
F1s: [0.97986577 0.99130435 0.984375   1.         0.98360656]
Precisions: [1.         1.         0.96923077 1.         0.96774194]
Recalls: [0.96052632 0.98275862 1.         1.         1.        ]


In [ ]:
df = pd.DataFrame(results)
df.to_csv('results/bert_multiclass2.csv', index=False)

In [11]:
df

,seed,batch_size,learning_rate,train_acc,train_precisions,train_recalls,train_f1s,max_memory_usage_train,max_vram_usage_train,total_train_time,...,val_precisions,val_recalls,val_f1s,test_acc,test_precisions,test_recalls,test_f1s,max_memory_usage_test,max_vram_usage_test,total_test_time
0,2,32,0.00002,0.996032,"[0.997093023255814, 1.0, 0.986013986013986, 1....","[0.9884726224783862, 0.9961832061068703, 0.996...","[0.9927641099855282, 0.9980879541108987, 0.991...",1361.660156,3806.438477,79.090446,...,"[0.9770114942528736, 0.9705882352941176, 0.986...","[0.9770114942528736, 1.0, 1.0, 0.9885057471264...","[0.9770114942528736, 0.9850746268656716, 0.993...",0.991018,"[1.0, 1.0, 0.9692307692307692, 1.0, 0.98360655...","[0.9605263157894737, 1.0, 1.0, 1.0, 1.0]","[0.9798657718120806, 1.0, 0.984375, 1.0, 0.991...",1378.070312,1766.021484,5.901150
1,3,32,0.00002,0.995370,"[0.997093023255814, 0.9923664122137404, 0.9894...","[0.9884726224783862, 0.9923664122137404, 0.996...","[0.9927641099855282, 0.9923664122137404, 0.992...",1382.355469,3820.313477,77.967063,...,"[0.9550561797752809, 0.9692307692307692, 0.985...","[0.9770114942528736, 0.9545454545454546, 0.957...","[0.9659090909090909, 0.9618320610687023, 0.971...",0.988024,"[1.0, 1.0, 0.9692307692307692, 1.0, 0.96774193...","[0.9605263157894737, 0.9827586206896551, 1.0, ...","[0.9798657718120806, 0.991304347826087, 0.9843...",1302.121094,1773.021484,5.292082
2,5,32,0.00002,0.988757,"[0.9854651162790697, 0.9923954372623575, 0.985...","[0.9769452449567724, 0.9961832061068703, 0.989...","[0.9811866859623734, 0.9942857142857143, 0.987...",1382.957031,3816.938477,78.274550,...,"[0.9438202247191011, 0.9848484848484849, 0.985...","[0.9655172413793104, 0.9848484848484849, 0.971...","[0.9545454545454546, 0.9848484848484849, 0.978...",0.988024,"[1.0, 1.0, 0.9692307692307692, 1.0, 0.96774193...","[0.9605263157894737, 0.9827586206896551, 1.0, ...","[0.9798657718120806, 0.991304347826087, 0.9843...",1302.347656,1779.521484,5.534412
